In [1]:
import pandas as pd
import numpy as np
import math
import csv

In [2]:
excel_file_name = '/Users/gabrieladaza/Desktop/mortality_data_test/fitting_mci_doc_cod/COD Coding Input template fixed.xlsx'
sub_sheet = 'DATA'

keywords_excel = '/Users/gabrieladaza/Desktop/mortality_data_test/fitting_mci_doc_cod/final_kw_v1.xlsx'
kw_sub_sheet = 'keywords'

In [3]:
df = pd.read_excel(excel_file_name, sheet_name="Sheet1", engine='openpyxl')

df.head()

,CASE_NUMBER,MANNER,CAUSE
0,1,"Suicide (e.g. hanging, knife/cutting instrumen...",NaN
1,2,"Illness (exclude AIDS-related deaths), specify...","Multi System Organ Failure, sepsis, low heart ..."
2,3,"Illness (exclude AIDS-related deaths), specify...",Renal Failure
3,4,Homicide (all years except 2011 and 2012),NaN
4,5,"Illness (exclude AIDS-related deaths), specify...","Stage 4 cancer, do not rescusitate"


In [4]:
df['MANNER'] = df['MANNER'].str.strip().str.lower()
df['CAUSE'] = df['CAUSE'].str.strip().str.lower()

In [5]:
df['ACCIDENT'] = False
df['NATURAL'] = False
df['HOMICIDE'] = False
df['SUICIDE'] = False
df['UNKNOWN'] = False
df['PENDING'] = False

In [6]:
df.loc[df['MANNER'] == 'accident', 'ACCIDENT'] = True
df.loc[df['MANNER'] == 'natural', 'NATURAL'] = True
df.loc[df['MANNER'] == 'homicide', 'HOMICIDE'] = True
df.loc[df['MANNER'] == 'suicide', 'SUICIDE'] = True
df.loc[df['MANNER'] == 'unknown', 'UNKNOWN'] = True
df.loc[df['MANNER'] == 'undetermined', 'UNKNOWN'] = True
df.loc[df['MANNER'] == 'pending', 'PENDING'] = True

In [7]:
kw_df = pd.read_excel(keywords_excel, sheet_name=kw_sub_sheet, engine='openpyxl')

od_keywords = kw_df['OVERDOSE KEYWORDS'].dropna().tolist()
heat_keywords = kw_df['HEAT KEYWORDS'].dropna().tolist()
cold_keywords = kw_df['COLD KEYWORDS'].dropna().tolist()
inj_keywords = kw_df['INJURY KEYWORDS'].dropna().tolist()
heart_keywords = kw_df['HEART KEYWORDS'].dropna().tolist()
illness_keywords = kw_df['ILLNESS KEYWORDS'].dropna().tolist()
undetermined_keywords = kw_df['UNDETERMINED'].dropna().tolist()
firearm_keywords = kw_df['FIREARM'].dropna().tolist()

In [8]:
df['CAUSE'] = df['CAUSE'].str.strip().str.upper()

df['PRELIM_OD'] = False
df['PRELIM_HEAT'] = False
df['PRELIM_COLD'] = False
df['PRELIM_OTHER_INJURY'] = False
df['PRELIM_HOMICIDE'] = False
df['PRELIM_SUICIDE'] = False
df['PRELIM_HEART'] = False
df['PRELIM_ILLNESS'] = False
df['PRELIM_FIREARM'] = False
df['PRELIM_UNKNOWN'] = False

In [9]:
for kw in od_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_OD'] = True

for kw in heat_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_HEAT'] = True

for kw in cold_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_COLD'] = True

for kw in inj_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_OTHER_INJURY'] = True

for kw in heart_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_HEART'] = True

for kw in illness_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_ILLNESS'] = True

for kw in firearm_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_FIREARM'] = True

for kw in undetermined_keywords:
    df.loc[df['CAUSE'].str.contains(kw, na=False), 'PRELIM_UNKNOWN'] = True

In [10]:
df['FINAL_ACC_OD'] = df['ACCIDENT'] & df['PRELIM_OD']
df['FINAL_ALL_OD'] = ~df['NATURAL'] & df['PRELIM_OD']

df['FINAL_HEAT'] = df['ACCIDENT'] & df['PRELIM_HEAT'] & ~df['PRELIM_OD']
df['FINAL_HEAT_UNK'] = (df['ACCIDENT'] | df['UNKNOWN']) & df['PRELIM_HEAT'] & ~df['PRELIM_OD']

df['FINAL_COLD'] = df['ACCIDENT'] & df['PRELIM_COLD'] & ~df['PRELIM_OD']
df['FINAL_COLD_UNK'] = (df['ACCIDENT'] | df['UNKNOWN']) & df['PRELIM_COLD'] & ~df['PRELIM_OD']

df['FINAL_OTHER_INJURY'] = df['ACCIDENT'] & df['PRELIM_OTHER_INJURY']
df['FINAL_OTHER_INJURY_UNK'] = (df['ACCIDENT'] | df['UNKNOWN']) & df['PRELIM_OTHER_INJURY']

df['FINAL_HOMICIDE'] = df['PRELIM_HOMICIDE']
df['FINAL_SUICIDE'] = df['PRELIM_SUICIDE']

df['FINAL_HEART'] = df['NATURAL'] & df['PRELIM_HEART']
df['FINAL_HEART_UNK'] = (df['NATURAL'] | df['UNKNOWN']) & df['PRELIM_HEART']

df['FINAL_ILLNESS'] = df['NATURAL'] & df['PRELIM_ILLNESS']
df['FINAL_ILLNESS_UNK'] = (df['NATURAL'] | df['UNKNOWN']) & df['PRELIM_ILLNESS']

df['FINAL_UNKNOWN'] = df['UNKNOWN']

df['FINAL_UNKNOWN_UNK'] = df['UNKNOWN']

In [11]:
df['MANUAL'] = ~(
    df['FINAL_ACC_OD'] |
    df['FINAL_HEAT'] |
    df['FINAL_COLD'] |
    df['FINAL_OTHER_INJURY'] |
    df['FINAL_HOMICIDE'] |
    df['FINAL_SUICIDE'] |
    df['FINAL_HEART'] |
    df['FINAL_ILLNESS'] |
    df['FINAL_UNKNOWN']
)

df[df['MANUAL']].head()

,CASE_NUMBER,MANNER,CAUSE,ACCIDENT,NATURAL,HOMICIDE,SUICIDE,UNKNOWN,PENDING,PRELIM_OD,...,FINAL_OTHER_INJURY_UNK,FINAL_HOMICIDE,FINAL_SUICIDE,FINAL_HEART,FINAL_HEART_UNK,FINAL_ILLNESS,FINAL_ILLNESS_UNK,FINAL_UNKNOWN,FINAL_UNKNOWN_UNK,MANUAL
0,1,"suicide (e.g. hanging, knife/cutting instrumen...",NaN,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,2,"illness (exclude aids-related deaths), specify...","MULTI SYSTEM ORGAN FAILURE, SEPSIS, LOW HEART ...",False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,3,"illness (exclude aids-related deaths), specify...",RENAL FAILURE,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,4,homicide (all years except 2011 and 2012),NaN,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,5,"illness (exclude aids-related deaths), specify...","STAGE 4 CANCER, DO NOT RESCUSITATE",False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [12]:
df['COD'] = 0

df.loc[df['FINAL_ACC_OD'], 'COD'] = 1
df.loc[df['FINAL_HEAT'], 'COD'] = 2
df.loc[df['FINAL_COLD'], 'COD'] = 3
df.loc[df['FINAL_OTHER_INJURY'], 'COD'] = 4
df.loc[df['FINAL_HOMICIDE'], 'COD'] = 5
df.loc[df['FINAL_SUICIDE'], 'COD'] = 6
df.loc[df['FINAL_HEART'], 'COD'] = 7
df.loc[df['FINAL_ILLNESS'], 'COD'] = 8
df.loc[df['FINAL_UNKNOWN'], 'COD'] = 9

In [13]:
df['COD'].value_counts()

COD
0    49183
9      500
Name: count, dtype: int64

In [14]:
df.head()

,CASE_NUMBER,MANNER,CAUSE,ACCIDENT,NATURAL,HOMICIDE,SUICIDE,UNKNOWN,PENDING,PRELIM_OD,...,FINAL_HOMICIDE,FINAL_SUICIDE,FINAL_HEART,FINAL_HEART_UNK,FINAL_ILLNESS,FINAL_ILLNESS_UNK,FINAL_UNKNOWN,FINAL_UNKNOWN_UNK,MANUAL,COD
0,1,"suicide (e.g. hanging, knife/cutting instrumen...",NaN,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,0
1,2,"illness (exclude aids-related deaths), specify...","MULTI SYSTEM ORGAN FAILURE, SEPSIS, LOW HEART ...",False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,0
2,3,"illness (exclude aids-related deaths), specify...",RENAL FAILURE,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,0
3,4,homicide (all years except 2011 and 2012),NaN,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,0
4,5,"illness (exclude aids-related deaths), specify...","STAGE 4 CANCER, DO NOT RESCUSITATE",False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,0


In [15]:
df.to_excel("final_cod_output.xlsx", index=False)